In [1]:
import numpy as np
from metavision_core.event_io.raw_reader import RawReader
from matplotlib import pyplot as plt
from scipy import sparse
from tqdm import tqdm
import os
import gc
import cv2
from skimage.transform import warp
from skimage.registration import optical_flow_tvl1, optical_flow_ilk
from skimage.registration import phase_cross_correlation
from parallel_merge import process_line_by_line

In [2]:
"""
BLOCK: Read events from trigger mode
"""
file_name = ""
raw_stream_x = RawReader("../data/grayscale_example/"+file_name+"y.raw", max_events=int(5e10))
output_folder = "../results/grayscale_example/"
os.makedirs(output_folder, exist_ok=True)
events_x = raw_stream_x.load_n_events(int(5e10-1e6))
external_triggers_x = raw_stream_x.get_ext_trigger_events()
positive_triggers_x = external_triggers_x[external_triggers_x['p']==1]

print("Number of events loaded:", len(events_x))
print(f"Total number of external triggers: {len(external_triggers_x)}")
print(f"Number of positive external triggers: {len(positive_triggers_x)}")
print("First 4 external triggers:", external_triggers_x[:4])
print("First 2 positive external triggers:", positive_triggers_x[:2])

trigger_times = np.array([trigger['t'] for trigger in positive_triggers_x[:25]])
trigger_differences = np.diff(trigger_times)
print("Differences between consecutive triggers:", trigger_differences)
print("Time for one line scan:", trigger_times[-1]-trigger_times[0])

pixel_per_mm = int(5910)
total_scan_area = 7 #mm
num_trigger_per_mm = 10
num_trigger_per_line = int((total_scan_area - 6) * num_trigger_per_mm + 1) #number of external triggers per line
mm_per_move = 0.1 #mm
sensor_width = 1000 #pixel
overlap_width = int(sensor_width - mm_per_move * pixel_per_mm)
fully_reconstruction_area = int(total_scan_area-6) #mm
merged_width = pixel_per_mm*total_scan_area + 640*2
trigger_interval = fully_reconstruction_area/(num_trigger_per_line-1) #mm

assert len(positive_triggers_x) == int(num_trigger_per_line * (fully_reconstruction_area / mm_per_move))
print(f"Overlap width: {overlap_width}")

lines_x = process_line_by_line(
    positive_triggers_t=positive_triggers_x['t'],
    events_t=events_x['t'],
    
    events_x_coord=events_x['y'], # y.raw
    events_y_coord=np.max(events_x['x'])-events_x['x'], # y.raw

    events_p=events_x['p'],
    total_lines=int(fully_reconstruction_area / mm_per_move),
    num_trigger_per_line=num_trigger_per_line,
    mm_per_move=mm_per_move,
    pixel_per_mm=pixel_per_mm,
    trigger_interval_x=trigger_interval,
    merged_height=merged_width, # y.raw
    merged_width=sensor_width, 
    manual_shift=0,
    axis="y") 


Number of events loaded: 1138289594
Total number of external triggers: 219
Number of positive external triggers: 110
First 4 external triggers: [(1, 2765139, 0) (0, 2774897, 0) (1, 2775146, 0) (0, 2785141, 0)]
First 2 positive external triggers: [(1, 2765139, 0) (1, 2775146, 0)]
Differences between consecutive triggers: [  10007   10249    9993    9754    9995   10007    9998    9997   10252
   10004 1983433   10006    9741   10253    9996    9751   10251    9997
    9752   10247   10008 1874183   10000   10252]
Time for one line scan: 4078126
Overlap width: 409


In [3]:
# average corse shift calculation
average_shift_odd = 0
average_shift_even = 0
average_overlap_width = 0
count_invalid_odd = 0
count_invalid_even = 0
for i in tqdm(range(0, len(lines_x)-1, 1)):
    line_x1 = lines_x[i][:, -overlap_width-1:-1].copy()
    line_x2 = lines_x[i+1][:, 0:overlap_width].copy()
    line_x1 = np.where(np.abs(line_x1) > 2, line_x1, 0)
    line_x2 = np.where(np.abs(line_x2) > 2, line_x2, 0)
    shift, error, diffphase = phase_cross_correlation(line_x1, line_x2, reference_mask=line_x1 != 0, moving_mask=line_x2 != 0)
    print(f"shift: {shift}")
    if np.abs(shift[0]) > 300:
        if i % 2 == 0:
            count_invalid_odd += 1
        else:
            count_invalid_even += 1
        continue
    if i % 2 == 0:
        average_shift_odd += shift[0]
    else:
        average_shift_even += shift[0]
    average_overlap_width += shift[1]
average_shift_odd /= len(lines_x) // 2 - count_invalid_odd  
average_shift_even /= len(lines_x) // 2 - count_invalid_even - (len(lines_x)-1) % 2
average_overlap_width /= len(lines_x) - count_invalid_odd - count_invalid_even
overlap_width = int(overlap_width - average_overlap_width)
print(f"average_shift_odd: {average_shift_odd}")
print(f"average_shift_even: {average_shift_even}")
print(f"average_overlap_width: {average_overlap_width}")
print(f"overlap_width: {overlap_width}")
print(f"count_invalid_odd: {count_invalid_odd}")
print(f"count_invalid_even: {count_invalid_even}")
assert overlap_width > 30

 11%|█         | 1/9 [00:02<00:19,  2.43s/it]

shift: [-87.  -5.]


 22%|██▏       | 2/9 [00:04<00:16,  2.41s/it]

shift: [87.  3.]


 33%|███▎      | 3/9 [00:07<00:14,  2.40s/it]

...

shift: [-87.   2.]


 89%|████████▉ | 8/9 [00:19<00:02,  2.39s/it]

shift: [90.  6.]


100%|██████████| 9/9 [00:21<00:00,  2.40s/it]

shift: [-86.   5.]
average_shift_odd: -87.6
average_shift_even: 87.5
average_overlap_width: 0.2
overlap_width: 408
count_invalid_odd: 0
count_invalid_even: 0


In [4]:
for i in tqdm(range(0, len(lines_x)-1, 2)):
    lines_x[i+1] = np.roll(lines_x[i+1], int(average_shift_odd), axis=0)

100%|██████████| 5/5 [00:00<00:00, 1479.26it/s]


In [5]:
# DEBUG: Plot all event lines
line_width_x = sensor_width

empty_events = np.zeros((int(merged_width - 6*pixel_per_mm), line_width_x*int(total_scan_area / mm_per_move)), dtype=np.int8)
for i, line_x in enumerate(lines_x):
    empty_events[:, line_width_x * i : line_width_x * (i + 1)] = line_x

# np.save(os.path.join(output_folder, f"all_events.npy"), empty_events)
plt.imsave(os.path.join(output_folder, f"all_events_y.png"), empty_events[::8, ::8], vmin=-5, vmax=5)
plt.imsave(os.path.join(output_folder, f"exapmle_events_y.png"), lines_x[5], vmin=-5, vmax=5)

In [6]:
del events_x
del raw_stream_x
del empty_events
gc.collect()

48008

In [7]:
merged_events_x = lines_x[0]

for i in tqdm(range(int(fully_reconstruction_area / mm_per_move - 1)), desc="Merging events"):
    line_index_2 = i + 1

    line_x1 = merged_events_x[:, -overlap_width-1:-1].copy()
    line_x2 = lines_x[line_index_2][:, 0:overlap_width].copy()
    line_x1 = np.where(np.abs(line_x1) > 2, line_x1, 0)
    line_x2 = np.where(np.abs(line_x2) > 2, line_x2, 0)

    line_x1 = np.clip(line_x1, -8, 8)
    line_x2 = np.clip(line_x2, -8, 8)

    shift = [0, 0]
        
    shifted_line_x2 = line_x2
    shifted_x2 = lines_x[line_index_2]

    line_x1 = line_x1 - 9
    shifted_line_x2 = shifted_line_x2 - 9
    line_x1 = np.where(line_x1 != -9, line_x1, 0)
    shifted_line_x2 = np.where(shifted_line_x2 != -9, shifted_line_x2, 0)
    line_x1 = line_x1.astype(np.uint8)
    shifted_line_x2 = shifted_line_x2.astype(np.uint8)

    shift, error, diffphase = phase_cross_correlation(line_x1, shifted_line_x2, upsample_factor=10, reference_mask=line_x1 != 0, moving_mask=shifted_line_x2 != 0)
    if np.abs(shift[0]) > 100:
        print(f"unreasonable shift_x: {shift}, resetting to 0")
        shift[0] = 0
    if np.abs(shift[1]) > 20:
        print(f"unreasonable shift_y: {shift}, resetting to 0")
        shift[1] = 0

    initial_flow = np.full((line_x1.shape[0], line_x1.shape[1], 2), [-shift[1], -shift[0]], dtype=np.float32)
    flow = cv2.calcOpticalFlowFarneback(
        line_x1, shifted_line_x2, initial_flow,
        pyr_scale=0.95, levels=60, winsize=50,
        iterations=32, poly_n=7, poly_sigma=1.5, flags=cv2.OPTFLOW_USE_INITIAL_FLOW
    )
    u = flow[:, :, 0]
    v = flow[:, :, 1]
    # v, u = optical_flow_ilk(line_x1, shifted_line_x2, radius=400, num_warp=8)
    avg_u = np.mean(u)
    y_shift = int(overlap_width + avg_u)
    
    shifted_x2 = shifted_x2[:, y_shift:]
    nr, nc = shifted_x2.shape
    row_coords, col_coords = np.meshgrid(np.arange(nr), np.arange(nc), indexing='ij')

    v_per_row = np.mean(v, axis=1)
    x = np.linspace(0, 1, nc-overlap_width)
    x = np.concatenate((x, np.ones(overlap_width)))
    tqdm.write(f"shift: {shift}, avg_u: {avg_u}, min: {v_per_row.min()}, max: {v_per_row.max()}, line_max: {lines_x[line_index_2].max()}")
    v_per_row = v_per_row[:, np.newaxis] * (1 - x)

    row_coords, col_coords = np.meshgrid(np.arange(shifted_x2.shape[0]), np.arange(shifted_x2.shape[1]), indexing='ij')
    warp_shifted_x2 = warp(shifted_x2, np.array([row_coords + v_per_row, col_coords]), order=0)

    merged_events_x = np.concatenate((merged_events_x, warp_shifted_x2), axis=1)

np.save(os.path.join(output_folder, f"merged_shifted_final_y.npy"), merged_events_x)
plt.imsave(os.path.join(output_folder, f"merged_shifted_final_y.png"), merged_events_x, vmin=-10, vmax=10)
print(merged_events_x.shape)

Merging events:  11%|█         | 1/9 [00:25<03:25, 25.65s/it]

shift: [ 0. -6.], avg_u: 0.003448174800723791, min: -11.832862854003906, max: 5.32643461227417, line_max: 21


Merging events:  22%|██▏       | 2/9 [00:51<02:58, 25.51s/it]

shift: [0. 2.], avg_u: -0.5326418280601501, min: -6.816703796386719, max: 6.183931350708008, line_max: 20


Merging events:  33%|███▎      | 3/9 [01:16<02:32, 25.40s/it]

...

shift: [ 1. -1.], avg_u: 2.70513653755188, min: -8.77993106842041, max: 8.583154678344727, line_max: 21


Merging events:  89%|████████▉ | 8/9 [03:23<00:25, 25.46s/it]

shift: [3. 5.], avg_u: 1.1201105117797852, min: -10.758694648742676, max: 12.947771072387695, line_max: 22


Merging events: 100%|██████████| 9/9 [03:48<00:00, 25.44s/it]


shift: [1. 3.], avg_u: 0.2544323801994324, min: -12.87077808380127, max: 10.115228652954102, line_max: 19
(7190, 6320)
